In [ ]:
using Pkg
Pkg.activate(".")
Pkg.develop(path="..")

isCuda = try
    success(`nvidia-smi`)
catch
    false
end
if isCuda
    println("CUDA is available, using GPU acceleration.")
    using CUDA
end

using bslLD
bslLD.greet()

if isCuda
    println("Setting backend to CUDA.")
    bslLD.use_cuda!()
else
    println("CUDA not available, using CPU.")
end



In [ ]:
grid =  bslLD.Grid([0.0,-6.0,-6.0],[10.0,6.0,6.0],[64,32,32],1, 1.0, 3)
simTime = bslLD.SimulationTime(0.025, 300.0, gyro_frequency=1.0)


# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)
initFuncx(x) = 1+ 0.0000001 * rand()
f = bslLD.Distribution(grid, 0.5,initFuncv=initFuncv, initFuncx=initFuncx);


In [ ]:
print(Array(grid.vaxes[1]))

In [ ]:
mutable struct Diag
    rho::Vector
end
Diag() = Diag([])

function diags!(diags, f, rho, Ex, grid, simTime)
    simTime.step % 1 == 0 || return
    push!(diags.rho, copy(rho.data[:]))
end

function stepStrang!(f, grid, simTime, diag)
    phase_start = simTime.phase
    Ω = simTime.gyro_frequency

    # V half-step at phase(t)
    sol = bslLD.solve_fields(bslLD.Moments(bslLD.compute_density(f, grid)), grid, bslLD.PoissonSolver(-0.001))
    simTime.fraction_dt = 0.5
    bslLD.advectV!(f, grid, simTime, sol.E)

    # X full-step at phase(t + dt/2)
    simTime.phase = phase_start + Ω * simTime.dt * 0.5
    simTime.fraction_dt = 1.0
    bslLD.advectX!(f, grid, simTime)

    rho = bslLD.compute_density(f, grid)

    # V half-step at phase(t + dt)
    simTime.phase = phase_start + Ω * simTime.dt
    sol = bslLD.solve_fields(bslLD.Moments(rho), grid, bslLD.PoissonSolver(-0.001))
    simTime.fraction_dt = 0.5
    bslLD.advectV!(f, grid, simTime, sol.E)
    simTime.fraction_dt = 1.0

    # restore so advance!() applies the correct full-step increment
    simTime.phase = phase_start

    diags!(diag, f, rho, sol.E[1], grid, simTime)
end


In [ ]:
bslLD.ProgressMeter.ijulia_behavior(:clear)

In [ ]:
diags = Diag()
while bslLD.continue_advection(simTime, true)
    stepStrang!(f, grid, simTime, diags)
    bslLD.advance!(simTime)
end

In [ ]:
using FFTW, DSP, Statistics, CairoMakie

In [ ]:
windowed.size

In [ ]:
locData = transpose(hcat(map(x-> x.-mean(x), Array.(diags.rho))...))

Nx, Ny = size(locData)


omega = fftfreq(size(locData, 1), 1) / simTime.dt*2*pi
k = (fftfreq(size(locData, 2))*length(grid.xaxes[1]) *2*pi/grid.max[1])[1:round(Int,Ny/2)]


w = kaiser(Ny, 3)

windowed = locData .* w'        # broadcast along second dim (1 × Ny)

nOmegaMax = round(Int,size(locData, 1) ÷ 4)

fig, ax, plt = heatmap(
    k,
    omega[1:nOmegaMax],
    transpose(log.(abs.(fft(windowed))[1:nOmegaMax, 1:round(Int, Ny/2)])),
        axis = (
        xlabel = "k",
        ylabel = "ω",
        title = "Fourier Interpolation")
)

lines!(
    ax,
    k,
    sqrt.(1000 .+ 1 .+ 3 .* k.^2),
    label = "√(ω_LH² + 1 + 3k²)",
    color = :red
)

limits!(
    ax,
    nothing, nothing,
    0, maximum(omega[1:nOmegaMax])
)

axislegend(ax, position = :rt)



fig